In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
train_df = pd.read_csv('train.csv')
train_df

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:

label_column_length_len = len(train_df['label'])
print(f"Length of 'label' column using len(): {label_column_length_len}")


# Using palette argument to color each bar differently
sns.countplot(x='label', data=train_df, palette='viridis')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def image_show(df, row_number, visualize=False, test_df=False):
    '''
        Parameters:
            df --> Pandas dataframe type
            row_number --> row index (For example: 0, 1, 2, 3, etc.)
            visualize --> If True then displays the raw data into 28 by 28 pixels image.
                          If False then returns '28 by 28 pixels reshaped image data' and 'corresponding label'
    '''
    if test_df:
        df_wantedimage = df.iloc[row_number, :].values.reshape(28, 28)
        df_wantedlabel = "N/A" # No label in test data
    else:
        df_wantedimage = df.iloc[row_number, 1:].values.reshape(28, 28)
        df_wantedlabel = df.iloc[row_number, 0]

    if visualize:
        ax = sns.heatmap(df_wantedimage, cmap="gray", cbar=False)
        ax.set_title(f"Label: {df_wantedlabel}")
        ax.set_xticks(range(0,28,5))
        ax.set_xticklabels(range(0, 28, 5))
        ax.set_yticks(range(0,28,5))
        ax.set_yticklabels(range(0, 28, 5))
        plt.show() # Removed this extra call
    else:
      if test_df:
        return df.iloc[row_number, :], None # Return image data and None for label
      else:
        return df.iloc[row_number, 1:], df.iloc[row_number, 0]

In [ ]:
image_show(train_df, 42, visualize=True)
image_show(train_df, 4242, visualize=True)

In [ ]:
def dataframe_formatter(df, isTest=False):
    """
    Formats a DataFrame containing image data for use in a machine learning model.

    :param df: The DataFrame containing image data.
    :param isTest: A boolean indicating whether the DataFrame is for testing data.
    :return: A tuple containing X (image data) and y (labels).
    """
    if isTest:
        X = df.values.reshape(len(df), 28, 28)
        y = None
    else:
        X = df.iloc[:, 1:].values.reshape(len(df), 28, 28)
        y = df.iloc[:, 0].values
    return X, y

In [ ]:
X, y = dataframe_formatter(train_df, isTest=False)
X.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train.shape, X_test.shape

In [ ]:
# Create a CNN with Keras from TensorFlow
# Your inputs should have the same size as your images (same as in the reshape function above)

from tensorflow.keras import models, layers

model = models.Sequential(  # Create the model with layers sequentially
    [
        layers.Conv2D(filters=20, kernel_size=(3, 3), activation='relu', input_shape=(28,28,1)),  # 2D convolutional layer with 20 filters, each 3x3, using ReLU activation, input shape is 28x28x1 (image dimensions)
        layers.MaxPooling2D(pool_size=(2, 2)),  # Max-pooling layer with a 2x2 window to reduce image size
        layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu'),  # 2D convolutional layer with 32 filters, each 3x3, using ReLU activation
        layers.MaxPooling2D(pool_size=(2, 2)),  # Max-pooling layer with a 2x2 window
        layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu'),  # 2D convolutional layer with 32 filters, each 2x2, using ReLU activation
        layers.MaxPooling2D(pool_size=(2, 2)),  # Max-pooling layer with a 2x2 window
        layers.Flatten(),  # Flatten layer to convert the 2D output to 1D
        layers.Dense(200, activation='relu'),  # Fully connected layer with 200 neurons and ReLU activation
        layers.Dropout(0,2),  # Dropout layer with a dropout rate of 0.2 for regularization to prevent overfitting
        layers.Dense(10, activation='sigmoid')  # Fully connected layer with 10 output units (classes) and sigmoid activation
    ]
)
model.summary()  # Summary of the model to verify its structure

In [ ]:
# Compile the model
# We specify the optimizer 'adam', which is a popular and efficient stochastic optimizer for neural networks.
# We use the loss function 'sparse_categorical_crossentropy', commonly used for multi-class classification with discrete labels.
# We also specify that we want to track the 'accuracy' metric during training.

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Early stopping in case of 3 consecutive bad epochs (if we're not learning effectively and there's a risk of overfitting)
from tensorflow.keras.callbacks import EarlyStopping
stop_training = EarlyStopping(
    monitor='val_loss',   # on surveille la loss de validation
    patience=3,           # 3 epochs sans amélioration = stop
    restore_best_weights=True  # on garde les poids du meilleur epoch
)

In [ ]:
# Train the model with 20 epochs
model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test), callbacks=[stop_training])

In [ ]:
test_df = pd.read_csv('test.csv')
test_df

In [ ]:
X_test, _ = dataframe_formatter(test_df, isTest=True)

In [ ]:
predictions = model.predict(X_test)

In [ ]:
image_show(test_df, 42, visualize=True, test_df=True)
print(predictions[42])
image_show(test_df, 4242, visualize=True, test_df=True)
print(predictions[4242])